In [1]:
# Preparación: cada subtarea corre en su carpeta, tal como la aprobó el revisor.
import os
from pathlib import Path
from IPython.display import Image, display

RAIZ = Path.cwd()

def mostrar_figuras(carpeta):
    for png in sorted(Path(carpeta).rglob("*.png")):
        display(Image(filename=str(png)))

# Entrega parcial (sin presupuesto para redactar)

## Configuración y corpus

## Parte 1 — Chunking

## Parte 2 — Índice (embeddings + kNN)

## Parte 3 — Generación con contexto

## Diagnóstico final

## Cierre

## Lo que faltó

- T3 (Parte 3 — generación con contexto y las tres pruebas de fuego): presupuesto agotado

### Código de la subtarea T1

In [2]:
# Subtarea T1: código aprobado (se ejecuta en trabajo/T1/)
os.chdir(RAIZ / 'trabajo' / 'T1')

# -*- coding: utf-8 -*-
# =============================================================================
# T1 — Configuración y Parte 1 — Chunking fijo con solapamiento
# MMIA 6013 · Semana 2 · Miércoles — "RAG mínimo en ~60 líneas"
# Pipeline completo: chunking → índice → retrieval → prompt con contexto → LLM.
# Esta subtarea monta la base (imports + corpus DOCS) y resuelve el Ejercicio
# 1.1: chunk_fijo(texto, tamano=300, overlap=80) sobre caracteres, construcción
# de chunks/origen FUERA del ejercicio, impresión de chunks con su documento de
# origen y comparación de los cortes entre overlap=80 y overlap=0.
# =============================================================================

# -----------------------------------------------------------------------------
# Configuración base — código dado por el profesor
# -----------------------------------------------------------------------------
import os
import json
import numpy as np

# -----------------------------------------------------------------------------
# Corpus: documentación interna de una empresa ficticia — código dado
# -----------------------------------------------------------------------------
DOCS = {
    "politica_vacaciones.md": """Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo acumulan 1.5 días de vacaciones por mes trabajado,
hasta un máximo de 18 días por año. Las vacaciones deben solicitarse con al menos
15 días de anticipación a través del portal interno. Los días no utilizados pueden
transferirse al año siguiente hasta un máximo de 5 días. Durante el primer año,
los días solo pueden tomarse después de superar el período de prueba de 3
meses.""",
    "politica_remoto.md": """Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permitido hasta 3 días por semana para todos los roles
excepto soporte de infraestructura on-site. Los días remotos se coordinan con el
líder de equipo. Para trabajar desde el exterior del país se requiere aprobación
de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año.""",
    "gastos.md": """Política de reembolso de gastos de NimbusSoft.
Los gastos de viaje se reembolsan presentando factura dentro de los 30 días
posteriores al gasto. El límite diario de alimentación en viajes es de 45 USD.
Los pasajes aéreos deben comprarse en clase económica salvo vuelos de más de
8 horas, donde se permite económica premium con aprobación del gerente de área.""",
}

TAMANO, OVERLAP = 300, 80  # parámetros del enunciado (valores por defecto)

# -----------------------------------------------------------------------------
# Ejercicio 1.1 — chunking fijo con solapamiento (SOLUCIÓN)
# -----------------------------------------------------------------------------
def chunk_fijo(texto, tamano=300, overlap=80):
    """Fragmenta `texto` en ventanas deslizantes de `tamano` caracteres con
    solapamiento `overlap` (todo medido en caracteres). Devuelve una lista
    de strings.

    - overlap=0 : los fragmentos particionan el texto (sin repetir caracteres).
    - overlap>0 : cada frontera repite los últimos `overlap` caracteres del
      fragmento anterior, de modo que una idea que cruce el corte aparece
      completa en al menos un fragmento (a costa de inflar el índice en
      C/(C-V): con 300/80 -> 300/220 = +36.36%).
    """
    if tamano <= 0:
        raise ValueError("tamano debe ser un entero positivo")
    if overlap < 0 or overlap >= tamano:
        raise ValueError("overlap debe cumplir 0 <= overlap < tamano")
    chunks = []
    inicio = 0
    n = len(texto)
    while inicio < n:
        fin = min(inicio + tamano, n)
        chunks.append(texto[inicio:fin])
        if fin >= n:
            break
        inicio = fin - overlap
    return chunks


# Helper interno de análisis (NO sustituye a chunk_fijo: solo añade las
# posiciones [inicio, fin) de cada chunk dentro de su documento para poder
# decir con exactitud qué idea queda cortada y en qué chunk).
def _chunk_fijo_con_spans(texto, tamano=300, overlap=80):
    spans = []
    inicio = 0
    n = len(texto)
    while inicio < n:
        fin = min(inicio + tamano, n)
        spans.append((texto[inicio:fin], int(inicio), int(fin)))
        if fin >= n:
            break
        inicio = fin - overlap
    return spans


# -----------------------------------------------------------------------------
# El corpus fragmentado. Vive FUERA del ejercicio porque las Partes 2 y 3 lo
# consumen: si estuviera dentro de la solución del 1.1, la versión de estudiante
# no lo tendría. (Código dado, con chunk_fijo ya implementado.)
# -----------------------------------------------------------------------------
chunks, origen = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto):  # tamano=300, overlap=80 (por defecto)
        chunks.append(c)
        origen.append(nombre)

for i, c in enumerate(chunks):
    print(f"[{i}] ({origen[i]}) {c[:70]}...")

# --- Repetir la fragmentación con overlap=0 para comparar los cortes --------
chunks0, origen0 = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto, tamano=300, overlap=0):
        chunks0.append(c)
        origen0.append(nombre)

print("\n--- Misma fragmentación con overlap=0 (comparación de cortes) ---")
for i, c in enumerate(chunks0):
    print(f"[{i}] ({origen0[i]}) {c[:70]}...")

# -----------------------------------------------------------------------------
# Análisis de los cortes: ¿algún chunk corta una idea?
# Idea de referencia: la de los días transferibles que señala el enunciado
# (la oración completa, tal como queda en el corpus con su salto de línea).
# -----------------------------------------------------------------------------
DOC_IDEA = "politica_vacaciones.md"
IDEA_TRANSFERENCIA = ("Los días no utilizados pueden\n"
                      "transferirse al año siguiente hasta un máximo de 5 días.")
FRASE_CLAVE = "hasta un máximo de 5 días"


def _offsets_globales(overlap):
    """Índice global en el que empiezan los chunks de cada documento."""
    offset, acum = {}, 0
    for nombre, texto in DOCS.items():
        offset[nombre] = acum
        acum += len(chunk_fijo(texto, TAMANO, overlap))
    return offset


def analizar_idea(doc, idea, tamano, overlap, offsets):
    """Clasifica cada chunk del documento respecto a `idea`:
    'completa' si el chunk la contiene entera; 'cortan' si solo la solapa en
    parte (la mutila). Usa las posiciones reales de inicio/fin de cada chunk."""
    texto = DOCS[doc]
    pos_ini = texto.find(idea)
    if pos_ini < 0:
        raise ValueError("La idea no se encontró en el documento")
    pos_fin = pos_ini + len(idea)
    cortan, completas = [], []
    for j, (_c, ini, fin) in enumerate(_chunk_fijo_con_spans(texto, tamano, overlap)):
        g = offsets[doc] + j
        if ini <= pos_ini and pos_fin <= fin:
            completas.append(int(g))
        elif ini < pos_fin and fin > pos_ini:
            cortan.append(int(g))
    return {
        "idea": idea,
        "documento": doc,
        "posicion_en_documento": [int(pos_ini), int(pos_fin)],
        "chunks_que_la_cortan": cortan,
        "chunks_donde_aparece_completa": completas,
        "cortada_en_algun_chunk": bool(cortan),
        "aparece_completa_en_algun_chunk": bool(completas),
    }


offset80 = _offsets_globales(OVERLAP)
offset0 = _offsets_globales(0)

idea80 = analizar_idea(DOC_IDEA, IDEA_TRANSFERENCIA, TAMANO, OVERLAP, offset80)
idea0 = analizar_idea(DOC_IDEA, IDEA_TRANSFERENCIA, TAMANO, 0, offset0)
frase80 = analizar_idea(DOC_IDEA, FRASE_CLAVE, TAMANO, OVERLAP, offset80)
frase0 = analizar_idea(DOC_IDEA, FRASE_CLAVE, TAMANO, 0, offset0)

# --- Conteos -----------------------------------------------------------------
num_chunks = int(len(chunks))
num_chunks0 = int(len(chunks0))
chunks_por_documento = {n: int(sum(1 for o in origen if o == n)) for n in DOCS}
chunks_por_documento0 = {n: int(sum(1 for o in origen0 if o == n)) for n in DOCS}

# --- Inflación del índice (costo del solapamiento) ---------------------------
chars_docs = int(sum(len(t) for t in DOCS.values()))
chars_chunks80 = int(sum(len(c) for c in chunks))
chars_chunks0 = int(sum(len(c) for c in chunks0))
inflacion_teorica_80 = float(TAMANO / (TAMANO - OVERLAP) - 1)  # C/(C-V) - 1
inflacion_teorica_0 = 0.0
inflacion_real_80 = float(chars_chunks80 / chars_docs - 1)
inflacion_real_0 = float(chars_chunks0 / chars_docs - 1)

# --- Detalle de los cortes (borde de cada chunk) ------------------------------
def detalle_cortes(overlap, offsets):
    filas = []
    for nombre, texto in DOCS.items():
        for j, (c, ini, fin) in enumerate(_chunk_fijo_con_spans(texto, TAMANO, overlap)):
            filas.append({
                "chunk": int(offsets[nombre] + j),
                "documento": nombre,
                "inicio_caracter": int(ini),
                "fin_caracter": int(fin),
                "empieza_con": c[:45],
                "termina_con": c[-45:],
            })
    return filas


cortes80 = detalle_cortes(OVERLAP, offset80)
cortes0 = detalle_cortes(0, offset0)

reconstruccion0 = {n: bool("".join(chunk_fijo(t, TAMANO, 0)) == t)
                   for n, t in DOCS.items()}
reconstruccion80 = {n: bool("".join(chunk_fijo(t, TAMANO, OVERLAP)) == t)
                    for n, t in DOCS.items()}

# --- Comparación overlap=80 vs overlap=0 --------------------------------------
def _fmt(xs):
    return "[" + ", ".join(str(x) for x in xs) + "]"


pos_frase = int(frase80["posicion_en_documento"][0])
comparacion_overlap_80_vs_0 = {
    "num_chunks_overlap80": num_chunks,
    "num_chunks_overlap0": num_chunks0,
    "idea_aparece_completa_overlap80": idea80["aparece_completa_en_algun_chunk"],
    "idea_aparece_completa_overlap0": idea0["aparece_completa_en_algun_chunk"],
    "chunks_que_cortan_la_idea_overlap80": idea80["chunks_que_la_cortan"],
    "chunks_que_cortan_la_idea_overlap0": idea0["chunks_que_la_cortan"],
    "inflacion_teorica_indice_overlap80": inflacion_teorica_80,
    "inflacion_teorica_indice_overlap0": inflacion_teorica_0,
    "inflacion_real_corpus_overlap80": inflacion_real_80,
    "inflacion_real_corpus_overlap0": inflacion_real_0,
    "lectura": (
        f"Con overlap=80 la idea de los días transferibles queda cortada en el/los "
        f"chunk(s) {_fmt(idea80['chunks_que_la_cortan'])} pero aparece COMPLETA en el/los "
        f"chunk(s) {_fmt(idea80['chunks_donde_aparece_completa'])}: el solapamiento la salva "
        f"(costo: inflación teórica del índice C/(C-V) = {TAMANO}/{TAMANO - OVERLAP} = "
        f"+{100 * inflacion_teorica_80:.2f}%). Con overlap=0 la misma idea queda cortada en "
        f"{_fmt(idea0['chunks_que_la_cortan'])} y NO aparece completa en ninguno: sin "
        f"solapamiento, una idea que cruza la frontera queda mutilada en los dos fragmentos. "
        f"La subfrase '{FRASE_CLAVE}' empieza en el carácter {pos_frase} del documento, "
        f"después del primer corte (carácter {TAMANO}), así que cae entera en el chunk 1 con "
        f"ambas configuraciones {_fmt(frase80['chunks_donde_aparece_completa'])} vs "
        f"{_fmt(frase0['chunks_donde_aparece_completa'])}; la oración completa de la idea, "
        f"en cambio, solo sobrevive con overlap=80."
    ),
}

# -----------------------------------------------------------------------------
# Cifras principales por pantalla
# -----------------------------------------------------------------------------
print("\n=== CIFRAS PRINCIPALES — T1 · Parte 1 (chunking) ===")
print(f"num_chunks (tamano=300, overlap=80): {num_chunks}")
print(f"chunks_por_documento: {chunks_por_documento}")
print(f"num_chunks_overlap0: {num_chunks0}")
print(f"chunks_por_documento_overlap0: {chunks_por_documento0}")

idea_txt = IDEA_TRANSFERENCIA.replace("\n", " ")
print(f"\nIdea analizada: \"{idea_txt}\"")
print(f"  overlap=80 -> cortada en chunks {idea80['chunks_que_la_cortan']} | "
      f"completa en chunks {idea80['chunks_donde_aparece_completa']}")
print(f"  overlap=0  -> cortada en chunks {idea0['chunks_que_la_cortan']} | "
      f"completa en chunks {idea0['chunks_donde_aparece_completa']}")
print(f"  subfrase '{FRASE_CLAVE}': completa en {frase80['chunks_donde_aparece_completa']} "
      f"(overlap=80) y en {frase0['chunks_donde_aparece_completa']} (overlap=0)")

g0 = offset80[DOC_IDEA]
print("\nObservación del corte en 'politica_vacaciones.md':")
print(f"  overlap=80 · chunk[{g0}] termina con: {chunks[g0][-45:]!r}")
print(f"  overlap=80 · chunk[{g0 + 1}] empieza con: {chunks[g0 + 1][:45]!r}")
print(f"  overlap=0  · chunk[{g0}] termina con: {chunks0[g0][-45:]!r}")
print(f"  overlap=0  · chunk[{g0 + 1}] empieza con: {chunks0[g0 + 1][:45]!r}")

print(f"\nInflación teórica del índice con 300/80: C/(C-V)-1 = "
      f"{inflacion_teorica_80:.6f} (+{100 * inflacion_teorica_80:.2f}%)")
print(f"Inflación real en este corpus: {inflacion_real_80:.6f} (overlap=80) vs "
      f"{inflacion_real_0:.6f} (overlap=0)")
print(f"Reconstrucción exacta del documento con overlap=0: {reconstruccion0}")
print("\nLectura de la comparación overlap=80 vs overlap=0:")
print(comparacion_overlap_80_vs_0["lectura"])

# -----------------------------------------------------------------------------
# Guardar TODAS las cifras en resultados.json (carpeta actual)
# -----------------------------------------------------------------------------
resultados = {
    "subtarea": "T1 — Configuración y Parte 1 — chunking fijo con solapamiento",
    "parametros_chunking": {
        "tamano_caracteres": int(TAMANO),
        "overlap_caracteres": int(OVERLAP),
        "overlap_comparacion": 0,
        "unidad": "caracteres",
    },
    "num_chunks": num_chunks,
    "chunks_por_documento": chunks_por_documento,
    "idea_cortada_overlap80": idea80,
    "idea_cortada_overlap0": idea0,
    "comparacion_overlap_80_vs_0": comparacion_overlap_80_vs_0,
    "idea_aparece_completa_overlap80": bool(idea80["aparece_completa_en_algun_chunk"]),
    "idea_aparece_completa_overlap0": bool(idea0["aparece_completa_en_algun_chunk"]),
    "frase_clave_hasta_max_5_dias_overlap80": frase80,
    "frase_clave_hasta_max_5_dias_overlap0": frase0,
    "num_chunks_overlap0": num_chunks0,
    "chunks_por_documento_overlap0": chunks_por_documento0,
    "longitud_documentos_caracteres": {n: int(len(t)) for n, t in DOCS.items()},
    "longitudes_chunks_overlap80": [int(len(c)) for c in chunks],
    "longitudes_chunks_overlap0": [int(len(c)) for c in chunks0],
    "caracteres_totales_docs": chars_docs,
    "caracteres_totales_chunks_overlap80": chars_chunks80,
    "caracteres_totales_chunks_overlap0": chars_chunks0,
    "inflacion_teorica_overlap80": inflacion_teorica_80,
    "inflacion_teorica_overlap0": inflacion_teorica_0,
    "inflacion_real_corpus_overlap80": inflacion_real_80,
    "inflacion_real_corpus_overlap0": inflacion_real_0,
    "cortes_overlap80": cortes80,
    "cortes_overlap0": cortes0,
    "chunks_overlap80": [{"chunk": int(i), "documento": origen[i], "texto": chunks[i]}
                         for i in range(len(chunks))],
    "chunks_overlap0": [{"chunk": int(i), "documento": origen0[i], "texto": chunks0[i]}
                        for i in range(len(chunks0))],
    "verificacion_reconstruccion_overlap0": reconstruccion0,
    "verificacion_reconstruccion_overlap80": reconstruccion80,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print(f"\n'resultados.json' guardado en la carpeta actual con {len(resultados)} claves.")

os.chdir(RAIZ)
mostrar_figuras(RAIZ / 'trabajo' / 'T1')

[0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo ...
[1] (politica_vacaciones.md) pación a través del portal interno. Los días no utilizados pueden
tran...
[2] (politica_remoto.md) Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permi...
[3] (politica_remoto.md) ra trabajar desde el exterior del país se requiere aprobación
de Recur...
[4] (gastos.md) Política de reembolso de gastos de NimbusSoft.
Los gastos de viaje se ...
[5] (gastos.md)  deben comprarse en clase económica salvo vuelos de más de
8 horas, do...

--- Misma fragmentación con overlap=0 (comparación de cortes) ---
[0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo ...
[1] (politica_vacaciones.md) l año siguiente hasta un máximo de 5 días. Durante el primer año,
los ...
[2] (politica_remoto.md) Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permi...
[3] (politica_remoto.md) s con 30 días de an

### Código de la subtarea T2

In [3]:
# Subtarea T2: código aprobado (se ejecuta en trabajo/T2/)
os.chdir(RAIZ / 'trabajo' / 'T2')

# -*- coding: utf-8 -*-
"""
T2 · Parte 2 — Índice (embeddings + kNN) y recuperar()
MMIA 6013 · Semana 2 · Miércoles — RAG mínimo en ~60 líneas.

Qué hace este script:
  1. Recupera (o reconstruye fielmente) los chunks de la Parte 1 (T1):
     chunk_fijo sobre caracteres, tamano=300 / overlap=80, sobre el corpus
     DOCS del enunciado.
  2. Ejecuta la celda dada del índice: sentence-transformers
     'paraphrase-multilingual-MiniLM-L12-v2' con embeddings normalizados
     (construye V y vec_consulta) o fallback TF-IDF normalizado si el modelo
     no está disponible.
  3. Mide el techo del modelo: cuenta los tokens del fragmento más largo con
     el tokenizador del propio modelo y los compara con max_seq_length,
     reportando si cabe entero o cuántos tokens se pierden en silencio.
  4. Ejercicio 2.1: recuperar(pregunta, k=3) -> [(score, idx_chunk), ...]
     ordenada de mayor a menor score, con esa firma exacta porque la Parte 3
     y el ejercicio 3.1 la llaman.

Salidas: resultados.json (todas las cifras, sin redondear) + print de las
cifras principales. Esta subtarea NO produce figura PNG.
"""

import json
import os

import numpy as np

# ---------------------------------------------------------------------------
# 0. Corpus del enunciado (mismo texto exacto que fragmenta la Parte 1)
# ---------------------------------------------------------------------------
DOCS = {
    "politica_vacaciones.md": (
        "Política de vacaciones de NimbusSoft.\n"
        "Los empleados a tiempo completo acumulan 1.5 días de vacaciones por mes trabajado,\n"
        "hasta un máximo de 18 días por año. Las vacaciones deben solicitarse con al menos\n"
        "15 días de anticipación a través del portal interno. Los días no utilizados pueden\n"
        "transferirse al año siguiente hasta un máximo de 5 días. Durante el primer año,\n"
        "los días solo pueden tomarse después de superar el período de prueba de 3\n"
        "meses."
    ),
    "politica_remoto.md": (
        "Política de trabajo remoto de NimbusSoft.\n"
        "El trabajo remoto está permitido hasta 3 días por semana para todos los roles\n"
        "excepto soporte de infraestructura on-site. Los días remotos se coordinan con el\n"
        "líder de equipo. Para trabajar desde el exterior del país se requiere aprobación\n"
        "de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año."
    ),
    "gastos.md": (
        "Política de reembolso de gastos de NimbusSoft.\n"
        "Los gastos de viaje se reembolsan presentando factura dentro de los 30 días\n"
        "posteriores al gasto. El límite diario de alimentación en viajes es de 45 USD.\n"
        "Los pasajes aéreos deben comprarse en clase económica salvo vuelos de más de\n"
        "8 horas, donde se permite económica premium con aprobación del gerente de área."
    ),
}

# ---------------------------------------------------------------------------
# 1. Chunks: textos de T1 si están guardados; si no, reconstrucción fiel
# ---------------------------------------------------------------------------
T1_PATH = os.path.join("entradas", "T1.json")
t1 = None
if os.path.exists(T1_PATH):
    with open(T1_PATH, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    print(f"entradas/T1.json leído. Claves: {sorted(t1.keys())}")
else:
    print("(no se encontró entradas/T1.json; se reconstruye todo desde el enunciado)")

# Parámetros de chunking declarados por T1 (por defecto, los del enunciado)
tamano, overlap = 300, 80
if isinstance(t1, dict) and isinstance(t1.get("parametros_chunking"), dict):
    pc = t1["parametros_chunking"]
    for clave in ("tamano", "tamano_caracteres", "chunk_size", "tam"):
        if isinstance(pc.get(clave), (int, float)):
            tamano = int(pc[clave])
            break
    for clave in ("overlap", "overlap_caracteres", "solape"):
        if isinstance(pc.get(clave), (int, float)):
            overlap = int(pc[clave])
            break


def _chunk_fijo(texto, tam, ovl, modo="corte"):
    """chunk_fijo de la Parte 1, sobre caracteres.
    modo='corte': al llegar al final se corta (chunk final parcial, estándar).
    modo='cola': sin corte; puede quedar un chunk final corto (variante)."""
    paso = tam - ovl
    if paso <= 0:
        paso = tam
    chunks, inicio, n = [], 0, len(texto)
    while inicio < n:
        chunks.append(texto[inicio:inicio + tam])
        if modo == "corte" and inicio + tam >= n:
            break
        inicio += paso
    return chunks


def _origen_de(chunk):
    for nombre, texto in DOCS.items():
        if chunk in texto:
            return nombre
    return "desconocido"


chunks, origen, fuente_chunks = None, None, None

if isinstance(t1, dict):
    # (a) lista plana de textos de chunks guardada por T1
    for clave in ("chunks", "textos_chunks", "chunks_overlap80", "chunks_textos"):
        v = t1.get(clave)
        if isinstance(v, list) and v and all(isinstance(x, str) for x in v):
            chunks = list(v)
            origen = [_origen_de(c) for c in chunks]
            fuente_chunks = f"textos leídos de entradas/T1.json ('{clave}')"
            break
    # (b) chunks agrupados por documento como listas de strings
    if chunks is None:
        cpd = t1.get("chunks_por_documento")
        if isinstance(cpd, dict) and cpd:
            valores = list(cpd.values())
            if all(isinstance(x, list) and x and isinstance(x[0], str) for x in valores):
                chunks, origen = [], []
                for nombre, lista in cpd.items():
                    chunks.extend(lista)
                    origen.extend([nombre] * len(lista))
                fuente_chunks = "textos leídos de entradas/T1.json ('chunks_por_documento')"

if chunks is None:
    # (c) reconstrucción con chunk_fijo; se elige la variante que cuadre con T1
    def _construir(modo):
        cs, og = [], []
        for nombre, texto in DOCS.items():
            partes = _chunk_fijo(texto, tamano, overlap, modo)
            cs.extend(partes)
            og.extend([nombre] * len(partes))
        return cs, og

    num_t1 = t1.get("num_chunks") if isinstance(t1, dict) else None
    longs_t1 = t1.get("longitudes_chunks_overlap80") if isinstance(t1, dict) else None

    def _cuadra(cand):
        cs, _ = cand
        if isinstance(num_t1, (int, float)) and len(cs) != int(num_t1):
            return False
        if (isinstance(longs_t1, list) and len(longs_t1) == len(cs)
                and all(isinstance(a, (int, float)) for a in longs_t1)):
            if not all(int(a) == len(c) for a, c in zip(longs_t1, cs)):
                return False
        return True

    cand_corte = _construir("corte")
    cand_cola = _construir("cola")
    if _cuadra(cand_corte):
        chunks, origen = cand_corte
        fuente_chunks = (f"reconstruidos con chunk_fijo(tamano={tamano}, overlap={overlap}) "
                         "sobre el corpus del enunciado (variante estándar, verificada contra T1)")
    elif _cuadra(cand_cola):
        chunks, origen = cand_cola
        fuente_chunks = (f"reconstruidos con chunk_fijo(tamano={tamano}, overlap={overlap}) "
                         "sobre el corpus del enunciado (variante con chunk final corto, verificada contra T1)")
    else:
        chunks, origen = cand_corte
        fuente_chunks = (f"reconstruidos con chunk_fijo(tamano={tamano}, overlap={overlap}) "
                         "sobre el corpus del enunciado (variante estándar; sin coincidencia exacta con T1)")

print(f"\nChunks ({len(chunks)}) — fuente: {fuente_chunks}")
for i, c in enumerate(chunks):
    print(f"[{i}] ({origen[i]}) {len(c)} car. · {c[:70].replace(chr(10), ' ')}...")

# Verificación explícita contra las cifras de T1
verificacion_t1 = {"t1_leido": bool(t1 is not None)}
if isinstance(t1, dict):
    if isinstance(t1.get("num_chunks"), (int, float)):
        verificacion_t1["num_chunks_T1"] = int(t1["num_chunks"])
        verificacion_t1["num_chunks_coincide"] = bool(len(chunks) == int(t1["num_chunks"]))
    lt1 = t1.get("longitudes_chunks_overlap80")
    if isinstance(lt1, list):
        if len(lt1) == len(chunks) and all(isinstance(a, (int, float)) for a in lt1):
            verificacion_t1["longitudes_coinciden"] = bool(
                all(int(a) == len(c) for a, c in zip(lt1, chunks)))
        else:
            verificacion_t1["longitudes_coinciden"] = None
print("Verificación con T1:", verificacion_t1)

# ---------------------------------------------------------------------------
# 2. Celda dada — índice: embeddings normalizados (o fallback TF-IDF)
# ---------------------------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer
    try:
        # local_files_only evita cualquier intento de descarga (si la versión lo soporta)
        _m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", local_files_only=True)
    except TypeError:
        _m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    def embed(ts):
        return np.asarray(_m.encode(ts, normalize_embeddings=True))

    MOTOR = "denso (sentence-transformers)"
    V = embed(chunks)

    def vec_consulta(q):
        return embed([q])[0]
except Exception:
    # El código dado captura solo ImportError; sin red/caché el modelo puede
    # fallar a cargar por otras razones: misma semántica de fallback.
    from sklearn.feature_extraction.text import TfidfVectorizer
    _tf = TfidfVectorizer().fit(chunks)  # se ajusta SOLO con el corpus indexado
    Vs = _tf.transform(chunks).toarray()
    V = Vs / (np.linalg.norm(Vs, axis=1, keepdims=True) + 1e-9)

    def vec_consulta(q):
        v = _tf.transform([q]).toarray()[0]
        return v / (np.linalg.norm(v) + 1e-9)

    MOTOR = "léxico (TF-IDF fallback)"

print("\nRetrieval:", MOTOR, "| índice:", V.shape)

# ---------------------------------------------------------------------------
# 3. El techo del modelo, medido (celda dada)
# ---------------------------------------------------------------------------
tope = None
tokens_por_chunk = None
idx_mas_largo = None
tokens_mas_largo = None
cabe_entero = None
tokens_perdidos = None
nota_techo = None

if MOTOR.startswith("denso"):
    tope = int(_m.max_seq_length)
    tokens_por_chunk = [int(len(_m.tokenizer(c)["input_ids"])) for c in chunks]
    idx_mas_largo = int(np.argmax(tokens_por_chunk))
    tokens_mas_largo = int(tokens_por_chunk[idx_mas_largo])
    cabe_entero = bool(tokens_mas_largo <= tope)
    tokens_perdidos = int(max(0, tokens_mas_largo - tope))
    print(f"\ntope del modelo: {tope} tokens")
    print(f"fragmento más largo: [{idx_mas_largo}] con {tokens_mas_largo} tokens "
          f"({len(chunks[idx_mas_largo])} caracteres)")
    print("cabe entero" if cabe_entero else
          f"NO cabe: se pierden {tokens_perdidos} tokens en silencio")
    nota_techo = (
        f"Medido con el tokenizador del propio modelo "
        f"(paraphrase-multilingual-MiniLM-L12-v2, max_seq_length={tope}). "
        + ("El fragmento más largo cabe entero: no se pierde ningún token."
           if cabe_entero else
           f"El fragmento más largo NO cabe: {tokens_perdidos} tokens se pierden "
           "en silencio (el modelo trunca sin avisar).")
    )
else:
    cabe_entero = True   # el fallback léxico no trunca: lee el fragmento entero
    tokens_perdidos = 0
    nota_techo = ("Sin sentence-transformers no hay modelo que truncar: el fallback TF-IDF "
                  "lee el fragmento entero (no hay truncado en tokens). El techo del MiniLM "
                  "multilingüe referenciado en el corpus del curso es 128 tokens "
                  "(fila embed_notebook_s2), pero no aplica a este motor léxico y aquí no "
                  "pudo medirse con el tokenizador del modelo.")
    print("\nSin sentence-transformers no hay modelo que truncar: el fallback TF-IDF lee "
          "el fragmento entero.")

# ---------------------------------------------------------------------------
# 4. Ejercicio 2.1 — recuperar(pregunta, k=3) → [(score, idx_chunk), ...]
# ---------------------------------------------------------------------------
def recuperar(pregunta, k=3):
    """Devuelve los k chunks más relevantes como [(score, idx_chunk), ...],
    ordenados de mayor a menor score, usando el índice V y vec_consulta.
    Firma exacta: la Parte 3 y el ejercicio 3.1 la llaman así."""
    q = vec_consulta(pregunta)
    scores = V @ q                        # coseno: filas de V y q normalizadas
    orden = np.argsort(-scores, kind="stable")[:k]
    return [(float(scores[i]), int(i)) for i in orden]


# Sanity check con las preguntas que usan la Parte 3 y el ejercicio 3.1
PREGUNTAS_DEMO = [
    ("¿puedo trabajar desde otro país?", "demo de armar_prompt (Parte 3)"),
    ("¿Cuántos días de vacaciones puedo transferir al año siguiente?", "transferir"),
    ("¿puedo trabajar desde el exterior?", "exterior"),
    ("¿puedo llevar a mi mascota a la oficina?", "mascotas (fuera del corpus)"),
    ("¿qué necesita aprobación del gerente?", "ambigua"),
]
demo_recuperar = {}
print("\nSanity check de recuperar(pregunta, k=3):")
for pregunta, etiqueta in PREGUNTAS_DEMO:
    res = recuperar(pregunta, k=3)
    demo_recuperar[pregunta] = {
        "etiqueta": etiqueta,
        "top_k": [{"score": s, "idx_chunk": i, "origen": origen[i],
                   "extracto": chunks[i][:70]} for s, i in res],
    }
    print(f"\n▸ {pregunta}  [{etiqueta}]")
    for score, i in res:
        print(f"   {score:.3f}  [{i}] ({origen[i]}) {chunks[i][:70].replace(chr(10), ' ')}...")

# ---------------------------------------------------------------------------
# 5. Guardar todas las cifras en resultados.json (sin redondear)
# ---------------------------------------------------------------------------
resultados = {
    "subtarea": "T2 · Parte 2 — Índice (embeddings + kNN) y recuperar()",
    "motor_retrieval": MOTOR,
    "forma_indice_V": [int(V.shape[0]), int(V.shape[1])],
    "tope_tokens_modelo": tope,
    "tokens_fragmento_mas_largo": tokens_mas_largo,
    "fragmento_cabe_entero": bool(cabe_entero),
    "tokens_perdidos_fragmento_mas_largo": int(tokens_perdidos) if tokens_perdidos is not None else None,
    "idx_fragmento_mas_largo": idx_mas_largo,
    "longitud_caracteres_fragmento_mas_largo": (int(len(chunks[idx_mas_largo]))
                                                if idx_mas_largo is not None else None),
    "tokens_por_chunk": tokens_por_chunk,
    "nota_techo": nota_techo,
    "num_chunks": int(len(chunks)),
    "origen_chunks": list(origen),
    "longitudes_chunks_caracteres": [int(len(c)) for c in chunks],
    "parametros_chunking_usados": {"tamano": int(tamano), "overlap": int(overlap)},
    "fuente_chunks": fuente_chunks,
    "verificacion_con_T1": verificacion_t1,
    "demo_recuperar_k3": demo_recuperar,
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print("\nresultados.json guardado en la carpeta actual.")

# ---------------------------------------------------------------------------
# 6. Cifras principales
# ---------------------------------------------------------------------------
print("\n=== Cifras principales — T2 · Parte 2 ===")
print("motor_retrieval:", MOTOR)
print("forma_indice_V:", [int(V.shape[0]), int(V.shape[1])])
print("tope_tokens_modelo:", tope)
print("tokens_fragmento_mas_largo:", tokens_mas_largo)
print("fragmento_cabe_entero:", cabe_entero)
if tope is not None:
    print("tokens_perdidos_fragmento_mas_largo:", tokens_perdidos)

os.chdir(RAIZ)
mostrar_figuras(RAIZ / 'trabajo' / 'T2')

entradas/T1.json leído. Claves: ['caracteres_totales_chunks_overlap0', 'caracteres_totales_chunks_overlap80', 'caracteres_totales_docs', 'chunks_overlap0', 'chunks_overlap80', 'chunks_por_documento', 'chunks_por_documento_overlap0', 'comparacion_overlap_80_vs_0', 'cortes_overlap0', 'cortes_overlap80', 'frase_clave_hasta_max_5_dias_overlap0', 'frase_clave_hasta_max_5_dias_overlap80', 'idea_aparece_completa_overlap0', 'idea_aparece_completa_overlap80', 'idea_cortada_overlap0', 'idea_cortada_overlap80', 'inflacion_real_corpus_overlap0', 'inflacion_real_corpus_overlap80', 'inflacion_teorica_overlap0', 'inflacion_teorica_overlap80', 'longitud_documentos_caracteres', 'longitudes_chunks_overlap0', 'longitudes_chunks_overlap80', 'num_chunks', 'num_chunks_overlap0', 'parametros_chunking', 'subtarea', 'verificacion_reconstruccion_overlap0', 'verificacion_reconstruccion_overlap80']

Chunks (6) — fuente: reconstruidos con chunk_fijo(tamano=300, overlap=80) sobre el corpus del enunciado (variante e


Retrieval: léxico (TF-IDF fallback) | índice: (6, 108)

Sin sentence-transformers no hay modelo que truncar: el fallback TF-IDF lee el fragmento entero.

Sanity check de recuperar(pregunta, k=3):

▸ ¿puedo trabajar desde otro país?  [demo de armar_prompt (Parte 3)]
   0.368  [3] (politica_remoto.md) ra trabajar desde el exterior del país se requiere aprobación de Recur...
   0.209  [2] (politica_remoto.md) Política de trabajo remoto de NimbusSoft. El trabajo remoto está permi...
   0.000  [0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft. Los empleados a tiempo completo ...

▸ ¿Cuántos días de vacaciones puedo transferir al año siguiente?  [transferir]
   0.499  [0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft. Los empleados a tiempo completo ...
   0.395  [1] (politica_vacaciones.md) pación a través del portal interno. Los días no utilizados pueden tran...
   0.223  [3] (politica_remoto.md) ra trabajar desde el exterior del país se requiere aprobación